# Activity 2 — Scraping economic data for Trinidad and Tobago

This notebook downloads five series into `raw/`, from the easiest source to the most work.
Every one of them is a source the Trinidad and Tobago nowcasting pipeline actually uses.

| § | Technique | Source | Series |
|---|---|---|---|
| 1 | API with a key → JSON | **FRED** (St. Louis Fed) | WTI oil price, exchange rates |
| 2 | HTML table → `BeautifulSoup` | **Central Bank of Trinidad and Tobago** data centre | new motor-vehicle sales |
| 3 | SDMX-XML → `ElementTree` | **Central Statistical Office** | quarterly GDP (the target) |
| 4 | Link harvesting + Excel, sheet matched by pattern | **CBTT** Economic Datapack | quarterly GDP by sector |
| 5 | Link harvesting + `melt` | **US Census Bureau** | US trade with Trinidad and Tobago |

**Offline mode.** `fetch()` keeps a copy of everything it downloads in `data/offline/`. If a
site does not respond, the cell uses that copy automatically. To run the whole notebook without
internet, set `OFFLINE = True` in the parameters cell.

Cells marked `<YOUR CODE HERE: ...>` are for you to complete. The solution is in
`sols/act2_scraping.ipynb`.

In [ ]:
# ===========================================================================
# Libraries, parameters and two helper functions.
# ===========================================================================
import os, re, json, warnings
from datetime import date
from io import BytesIO, StringIO
from urllib.parse import urljoin
import xml.etree.ElementTree as ET

import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup

warnings.filterwarnings("ignore")
requests.packages.urllib3.disable_warnings()   # we deliberately skip some SSL checks below

# ---- parameters ------------------------------------------------------------
OFFLINE      = False                 # True = never go online; use the copies in data/offline/
FRED_API_KEY = ""                    # your own key: https://fredaccount.stlouisfed.org/apikeys
FRED_API_KEY = FRED_API_KEY or os.environ.get("FRED_API_KEY", "")

PATH_RAW     = os.path.join("..", "raw")             # downloaded CSVs go here
PATH_OFFLINE = os.path.join("..", "data", "offline") # copies for offline mode
SOURCES_CSV  = os.path.join(PATH_RAW, "_sources.csv")
os.makedirs(PATH_RAW, exist_ok=True)
os.makedirs(PATH_OFFLINE, exist_ok=True)

# ---- the browser disguise --------------------------------------------------
# Many servers reject requests that do not look like a browser. Sending a
# User-Agent string is the single most useful line in this whole notebook.
hdr = {"User-Agent": ("Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                      "(KHTML, like Gecko) Chrome/120.0 Safari/537.36")}

MODE = {}   # copy -> ("online" | "offline copy", date), for the sources log


def fetch(url, copy, timeout=60, **kw):
    # Request `url` and return its content (bytes).
    # If it answers, keep a copy in data/offline/<copy>. If it does not answer,
    # or if OFFLINE = True, return that copy instead.
    path = os.path.join(PATH_OFFLINE, copy)
    if not OFFLINE:
        try:
            try:
                r = requests.get(url, headers=hdr, timeout=timeout, **kw)
            except requests.exceptions.SSLError:
                # some government sites run expired or incomplete certificates
                r = requests.get(url, headers=hdr, timeout=timeout, verify=False, **kw)
            print(f"  {r.status_code} | {r.headers.get('content-type', '')[:45]} | {len(r.content):,} bytes")
            if r.ok:
                with open(path, "wb") as f:
                    f.write(r.content)
                MODE[copy] = ("online", date.today().isoformat())
                return r.content
        except requests.exceptions.RequestException as e:
            print(f"  no answer: {type(e).__name__}")
    if not os.path.exists(path):
        raise FileNotFoundError(f"No offline copy of {copy} in {PATH_OFFLINE}")
    print(f"  -> using the offline copy {copy}")
    MODE[copy] = ("offline copy", date.fromtimestamp(os.path.getmtime(path)).isoformat())
    with open(path, "rb") as f:
        return f.read()


def register(file, source, url, df, copy):
    # Note in raw/_sources.csv where a file came from and what it holds.
    s = df.dropna(how="all")
    row = pd.DataFrame([{
        "file": file, "source": source, "url": url,
        "downloaded": MODE[copy][1], "mode": MODE[copy][0],
        "columns": df.shape[1], "from": s.index.min().date(), "to": s.index.max().date()}])
    if os.path.exists(SOURCES_CSV):
        old = pd.read_csv(SOURCES_CSV)
        row = pd.concat([old[old["file"] != file], row], ignore_index=True)
    row.to_csv(SOURCES_CSV, index=False)
    print(f"  logged in _sources.csv: {file} | {row.iloc[-1]['from']} to {row.iloc[-1]['to']}")


print("OFFLINE =", OFFLINE, "| FRED key set:", bool(FRED_API_KEY))

---
# § 1 — FRED: an API with a key

**Source shape:** the agency wants you to have the data and built a front door.

An API answers with structured data (JSON): no HTML to walk, no Excel to tidy. You ask for
**one series** by **its code** and get back dates and values. Other examples: the
[World Bank](https://datahelpdesk.worldbank.org/knowledgebase/topics/125589) and the
[IMF](https://data.imf.org/).

FRED needs a **free, personal key** (<https://fredaccount.stlouisfed.org/apikeys>). Paste it in
`FRED_API_KEY` in the parameters cell, or better, set it once as an environment variable.
**Never share a key or push it to GitHub.** Without a key the cell uses the offline copies.

The pipeline pulls several FRED series. We start with two:

| code | series |
|---|---|
| `POILWTIUSDM` | WTI crude oil price, US$ per barrel, monthly |
| `EXCAUS` | Canadian dollars per US dollar, monthly |

In [ ]:
FRED_URL = "https://api.stlouisfed.org/fred/series/observations"
series = ["POILWTIUSDM", "EXCAUS"]

fred = pd.DataFrame()
for code in series:
    print(code)
    params = {"series_id": code, "api_key": FRED_API_KEY, "file_type": "json"}
    if FRED_API_KEY:
        content = fetch(FRED_URL, f"fred_{code}.json", params=params)
    else:
        _was, OFFLINE = OFFLINE, True          # no key: go straight to the copy
        content = fetch(FRED_URL, f"fred_{code}.json")
        OFFLINE = _was

    obs = json.loads(content)["observations"]          # a list of {"date": ..., "value": ...}
    dfi = pd.DataFrame(obs)[["date", "value"]]
    dfi["date"] = pd.to_datetime(dfi["date"])
    dfi = dfi.set_index("date").rename(columns={"value": f"fred_{code}"})
    dfi = dfi.apply(pd.to_numeric, errors="coerce")     # FRED writes "." for missing
    fred = fred.join(dfi, how="outer") if not fred.empty else dfi

fred.tail()

**COMPLETE**: the pipeline also uses the US dollar–euro rate, `DEXUSEU` (daily). Add it to
the list and re-run the loop. Then plot the three series since 2015.

In [ ]:
series = ["POILWTIUSDM", "EXCAUS", <YOUR CODE HERE: add the US$/euro code>]

fred = pd.DataFrame()
for code in series:
    params = {"series_id": code, "api_key": FRED_API_KEY, "file_type": "json"}
    if FRED_API_KEY:
        content = fetch(FRED_URL, f"fred_{code}.json", params=params)
    else:
        _was, OFFLINE = OFFLINE, True
        content = fetch(FRED_URL, f"fred_{code}.json")
        OFFLINE = _was
    dfi = pd.DataFrame(json.loads(content)["observations"])[["date", "value"]]
    dfi["date"] = pd.to_datetime(dfi["date"])
    dfi = dfi.set_index("date").rename(columns={"value": f"fred_{code}"}).apply(pd.to_numeric, errors="coerce")
    fred = fred.join(dfi, how="outer") if not fred.empty else dfi

fred.to_csv(os.path.join(PATH_RAW, "fred_data.csv"))
register("fred_data.csv", "FRED", FRED_URL, fred, f"fred_{series[0]}.json")

# Each series has its own frequency: plot each one without its gaps
fig, axes = plt.subplots(1, len(series), figsize=(14, 3))
for ax, col in zip(axes, fred.columns):
    fred[col].loc["2015":].dropna().plot(ax=ax, title=col)
plt.tight_layout(); plt.show()

---
# § 2 — An HTML table: the Central Bank data centre

**Source shape:** the numbers are in a `<table>` on a web page.

The Central Bank of Trinidad and Tobago publishes most of its statistics in its
[data centre](https://www.central-bank.org.tt/statistics/data-centre/). Each table is a web page,
and the numbers sit in a `<table id="table_1">`.

### Step 1 — look before you code

Open the page for **new motor-vehicle sales** in your browser:

<https://www.central-bank.org.tt/data-center-block/?title=output-new-motor-vehicle-sales-monthly&id=153>

Right-click the table → **Inspect**. Find the `<table>` tag and its `id`. Note that the header
cells are `<th>` inside `<thead>`, and the data rows are `<tr>` inside `<tbody>`.

### Step 2 — request the page

In [ ]:
CBTT_URL = "https://www.central-bank.org.tt/data-center-block/?title=output-new-motor-vehicle-sales-monthly&id=153"

html = fetch(CBTT_URL, "cbtt_vehicle_sales.html")

# What arrives is text: the same HTML you saw with "Inspect"
print(html[:300].decode("utf-8", errors="replace"))

### Step 3 — find the table and read it

`BeautifulSoup` turns the HTML into a tree you can search. `select_one("table#table_1")` uses a
CSS selector: *the `<table>` whose `id` is `table_1`*.

In [ ]:
soup  = BeautifulSoup(html, "html.parser")
table = soup.select_one("table#table_1")
if table is None:
    raise ValueError("Could not find table_1 in the page: has the layout changed?")

# Header: the text of every <th> under <thead>
cols = [th.get_text(strip=True) for th in table.select("thead th")]

# Rows: the text of every <td> in every <tr> under <tbody>
rows = []
for tr in table.select("tbody tr"):
    tds = [td.get_text(strip=True) for td in tr.select("td")]
    if tds:
        rows.append(tds)

df = pd.DataFrame(rows, columns=cols)
print(cols)
df.head()

Everything arrived as **text**. Two things to fix:

* **dates** are written day first (`31/01/2024`) — tell pandas with `dayfirst=True`;
* **numbers** are text, and the bank writes `n.d.` / `n.a.` for missing values.
  `errors="coerce"` turns anything that is not a number into `NaN` instead of raising.

**COMPLETE** the conversion of the value columns.

In [ ]:
df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")

for c in df.columns[1:]:
    df[c] = <YOUR CODE HERE: convert column c to numbers, missing markers become NaN>

df = df.set_index("Date").sort_index().dropna(how="all")
df.dtypes

### Step 4 — tidy the column names

Good practice everywhere: lowercase, no punctuation, words joined by underscores, and a prefix
that says where the series came from. `Total New Motor Vehicles Sold` becomes
`veh_total_new_motor_vehicles_sold` — something you can type without quoting.

In [ ]:
def clean_col(c):
    c = c.lower()
    c = re.sub(r"\(([^)]*)\)", r" \1", c)     # "(commercial)" -> " commercial"
    c = re.sub(r"[^a-z0-9\s_-]", "", c)       # drop special characters
    c = re.sub(r"[\s\-]+", "_", c)            # spaces and hyphens -> _
    return "veh_" + re.sub(r"_+", "_", c).strip("_")

df.columns = [clean_col(c) for c in df.columns]
df.index.name = "date"

df.to_csv(os.path.join(PATH_RAW, "cbtt_vehicles_sales.csv"))
register("cbtt_vehicles_sales.csv", "CBTT", CBTT_URL, df, "cbtt_vehicle_sales.html")

df.loc["2015":].plot(figsize=(10, 3.5), title="New motor-vehicle sales, units per month (CBTT)")
plt.show()

### The one-line alternative: `pd.read_html`

When a page holds a clean `<table>`, `pd.read_html` finds every table and returns a **list** of
DataFrames. It is shorter, but you lose control: you cannot pick the table by its `id`, and
header rows are guessed. Try it on the same page:

In [ ]:
tables = pd.read_html(StringIO(html.decode("utf-8", errors="replace")))
print(f"{len(tables)} table(s) found")
tables[0].head()

**COMPLETE — your turn.** The data centre uses the **same layout for every table**, so the code
above works for any of them. Download **motor-vehicle registrations** (id 155):

<https://www.central-bank.org.tt/data-center-block/?title=output-motor-vehicle-registration-monthly&id=155>

Write a function `cbtt_table(url, copy)` that returns the table as a DataFrame with a date index,
and call it on this page.

In [ ]:
def cbtt_table(url, copy):
    html  = fetch(url, copy)
    table = BeautifulSoup(html, "html.parser").select_one("table#table_1")
    cols  = [th.get_text(strip=True) for th in table.select("thead th")]
    rows  = [[td.get_text(strip=True) for td in tr.select("td")] for tr in table.select("tbody tr")]
    df    = pd.DataFrame([r for r in rows if r], columns=cols)
    df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")
    for c in df.columns[1:]:
        df[c] = pd.to_numeric(df[c].replace({"n.d.": np.nan, "n.a.": np.nan}), errors="coerce")
    return df.set_index("Date").sort_index().dropna(how="all")


REG_URL = "https://www.central-bank.org.tt/data-center-block/?title=output-motor-vehicle-registration-monthly&id=155"
reg = <YOUR CODE HERE: call cbtt_table on REG_URL, offline copy "cbtt_vehicle_registration.html">
reg.tail()

---
# § 3 — SDMX-XML: quarterly GDP from the Central Statistical Office

**Source shape:** a machine-readable statistical exchange format.

The CSO publishes its headline tables under the IMF's **e-GDDS** standard, as SDMX-XML files.
SDMX is *designed* for machines, and its structure is always the same:

```xml
<Series INDICATOR="NGDP_PA_R_XDC" FREQ="Q" ...>
    <Obs TIME_PERIOD="2025-Q4" OBS_VALUE="37703.6"/>
    <Obs TIME_PERIOD="2026-Q1" OBS_VALUE="34645.1"/>
</Series>
```

So you walk the `<Series>` elements and read the attributes off each `<Obs>`.

* `.//{*}Series` is XPath for *every `<Series>`, at any depth, in any namespace*. SDMX files
  declare XML namespaces, and without the `{*}` wildcard `findall` finds nothing.
* `NGDP_PA_R_XDC` is real GDP, production approach — **the series we nowcast** (`RGDP0000`).

In [ ]:
CSO_GDP_URL = "https://cso.gov.tt/wp-content/uploads/SDMX/NAG_TTO.xml"

content = fetch(CSO_GDP_URL, "cso_NAG_TTO.xml")
root = ET.fromstring(content)

rows = []
for s in root.findall(".//{*}Series"):            # every <Series>, any depth, any namespace
    indicator = s.attrib.get("INDICATOR")
    for obs in s.findall("{*}Obs"):               # its <Obs> children
        rows.append({"date": obs.attrib.get("TIME_PERIOD"),
                     "indicator": indicator,
                     "value": pd.to_numeric(obs.attrib.get("OBS_VALUE"), errors="coerce")})

long = pd.DataFrame(rows)
print(f"{long['indicator'].nunique()} indicators, {len(long):,} observations")
long.head()

### Dates: quarter labels → the project's convention

The file mixes frequencies; keep the quarterly labels (`2025-Q4`). A `PeriodIndex` understands
them. We then stamp each quarter on the **first day of its last month** (2025Q4 → 2025-12-01),
the convention every pipeline in this project uses.

In [ ]:
q = long[long["date"].str.match(r"^\d{4}-Q[1-4]$")].copy()

per = pd.PeriodIndex(q["date"].str.replace("-", ""), freq="Q")   # "2025-Q4" -> 2025Q4
q["date"] = per.asfreq("M", "end").to_timestamp()                 # -> 2025-12-01

gdp = q.pivot(index="date", columns="indicator", values="value")  # long -> wide
gdp.to_csv(os.path.join(PATH_RAW, "cbtt_gdp.csv"))
register("cbtt_gdp.csv", "CSO (SDMX)", CSO_GDP_URL, gdp, "cso_NAG_TTO.xml")
gdp[["NGDP_PA_R_XDC"]].tail()

### Always check against something you know

`../data/gdp.csv` (Activity 1) is the same series, built by the project pipeline. If the two
disagree, the parsing is wrong — or the CSO has revised the data.

In [ ]:
known = pd.read_csv("../data/gdp.csv", index_col="date", parse_dates=True)["rgdp"]
check = pd.concat([gdp["NGDP_PA_R_XDC"].rename("scraped"), known.rename("pipeline")], axis=1).dropna()
check["diff_%"] = 100 * (check["scraped"] / check["pipeline"] - 1)
print(f"{len(check)} common quarters, largest gap {check['diff_%'].abs().max():.2f} %")
check.tail()

**COMPLETE.** The CSO publishes the consumer price index the same way:
`https://cso.gov.tt/wp-content/uploads/SDMX/CPI_TTO.xml` (monthly). Wrap the parsing above in a
function `read_sdmx(content)` that returns the long table, and use it to load the CPI.
Monthly labels look like `2026-06`, which `pd.to_datetime` reads directly.

In [ ]:
def read_sdmx(content):
    root = ET.fromstring(content)
    rows = []
    for s in root.findall(".//{*}Series"):
        ind = s.attrib.get("INDICATOR")
        for obs in s.findall("{*}Obs"):
            rows.append({"date": obs.attrib.get("TIME_PERIOD"), "indicator": ind,
                         "value": pd.to_numeric(obs.attrib.get("OBS_VALUE"), errors="coerce")})
    return pd.DataFrame(rows)


CSO_CPI_URL = "https://cso.gov.tt/wp-content/uploads/SDMX/CPI_TTO.xml"
cpi = <YOUR CODE HERE: fetch the CPI file (copy "cso_CPI_TTO.xml") and parse it with read_sdmx>
cpi["date"] = pd.to_datetime(cpi["date"])
cpi = cpi.pivot(index="date", columns="indicator", values="value")
cpi.to_csv(os.path.join(PATH_RAW, "cbtt_cpi.csv"))
cpi.iloc[-3:, :5]

---
# § 4 — Link harvesting and an Excel file: the CBTT Economic Datapack

**Source shape:** the data file exists, but its URL changes with every release.

The Central Bank's **Economic Datapack** is one workbook with most of its statistics. A new file
is posted every release, with a new name. You cannot hardcode the link. Instead, scrape the
*page*, collect every `<a href="...">`, and keep the one you want. The page layout is far more
stable than the file names.

In [ ]:
DATAPACK_PAGE = ("https://www.central-bank.org.tt/resources-category/"
                 "publications-and-research/all-reports/economic-datapack-report/")

page  = fetch(DATAPACK_PAGE, "cbtt_datapack_page.html")
soup  = BeautifulSoup(page, "html.parser")

links = sorted({urljoin(DATAPACK_PAGE, a["href"])          # urljoin fixes relative links
                for a in soup.find_all("a", href=True)
                if a["href"].lower().endswith((".xlsm", ".xlsx"))})
print(f"{len(links)} workbook link(s)")
for l in links[-5:]:
    print("  ", l)

**COMPLETE**: pick the latest workbook. The links are sorted, and the file names carry the
release date, so the latest is the last one.

In [ ]:
latest = <YOUR CODE HERE: the last link in the sorted list>
print("latest datapack:", latest)

xls = pd.ExcelFile(BytesIO(fetch(latest, "cbtt_datapack.xlsm", timeout=120)))
print(xls.sheet_names[:15])

`pd.ExcelFile` opens the workbook *without* reading every sheet, so you can look at
`.sheet_names` first and pick. **Match the sheet name with a pattern rather than hardcoding it**:
hardcoded sheet names are the most common way these scripts break when a new release comes out.

The quarterly real-activity sheet has a **three-row header** (group, sector, sub-sector) above
the data, with the first column holding the dates. Read it with `header=None` to see it raw.

In [ ]:
sheet = [s for s in xls.sheet_names if "DomesticAct_Quarterly" in s][0]
raw = pd.read_excel(xls, sheet_name=sheet, header=None, skiprows=8)
raw.iloc[:6, :6]

In [ ]:
head = raw.iloc[:3].astype(str)
group = head.iloc[0].replace("nan", np.nan).ffill()          # merged cells: fill the group label right

# keep the columns under the GDP group
qgdp = [c for c in raw.columns if re.search("Gross Domestic Product|QGDP", str(group[c]), re.I)]

def name(c):
    parts = [head.iloc[i, c] for i in (1, 2) if head.iloc[i, c].lower() not in ("nan", "", "of which:")]
    s = re.sub(r"\(.*?\)", "", " ".join(parts)).lower()
    s = re.sub(r"[^a-z0-9]+", "_", s).strip("_")
    return "qgdp_" + (s or f"col{c}")

dp = raw.iloc[3:, [0] + qgdp].copy()
dp.columns = ["date"] + [name(c) for c in qgdp]
dp["date"] = pd.to_datetime(dp["date"], errors="coerce")
dp = dp.dropna(subset=["date"]).set_index("date").apply(pd.to_numeric, errors="coerce").dropna(how="all")

dp.to_csv(os.path.join(PATH_RAW, "cbtt_gdp_datapack.csv"))
register("cbtt_gdp_datapack.csv", "CBTT datapack", latest, dp, "cbtt_datapack.xlsm")
dp.tail()

---
# § 5 — Link harvesting + reshaping: US trade with Trinidad and Tobago

The United States is Trinidad and Tobago's largest trading partner, and the US Census Bureau
publishes monthly trade **by country** — far sooner than the national trade statistics. The
spreadsheet link is on the country page; harvest it the same way.

The file arrives in the shape agencies love and analysts hate: **one row per country and year,
one column per month and flow** (`IJAN`, `IFEB`, ..., `EJAN`, ...: *I*mports and *E*xports).
We want a date index down the rows. The tool for that is `melt` (wide → long), then `pivot`
(long → wide, the right way round).

In [ ]:
CENSUS_PAGE = "https://www.census.gov/foreign-trade/balance/c2360.html"   # 2360 = Trinidad and Tobago

page  = fetch(CENSUS_PAGE, "census_c2360.html")
links = [a.get("href") for a in BeautifulSoup(page, "html.parser")("a") if a.get("href")]
xls_links = [l for l in links if "xls" in l.lower()]
print(xls_links)

trade = pd.read_excel(BytesIO(fetch(urljoin(CENSUS_PAGE, xls_links[0]), "census_country.xlsx", timeout=120)))
trade = trade[trade["CTYNAME"].str.contains("Trinidad and Tobago", na=False)]
trade.iloc[:3, :8]

**COMPLETE** the reshape:

* `melt` with `id_vars="year"`: every month column becomes a row;
* the first letter of the old column name is the flow, the rest is the month.

In [ ]:
trade = trade.drop(columns=["CTY_CODE", "CTYNAME", "IYR", "EYR"], errors="ignore")
trade.columns = trade.columns.str.lower()

long = <YOUR CODE HERE: melt trade to long format: id_vars="year", var_name="month", value_name="trade">
long["flow"]  = long["month"].str[0].map({"i": "USA_imp_TTO", "e": "USA_exp_TTO"})
long["month"] = long["month"].str[1:]
long["date"]  = pd.to_datetime(long["year"].astype(str) + long["month"], format="%Y%b") + pd.offsets.MonthEnd(0)

us = long.pivot(index="date", columns="flow", values="trade")
us = us[us["USA_exp_TTO"] != 0]            # months not yet published come as zeros
us.to_csv(os.path.join(PATH_RAW, "uscensus_trade.csv"))
register("uscensus_trade.csv", "US Census", CENSUS_PAGE, us, "census_country.xlsx")

us.loc["2015":].plot(figsize=(10, 3.5), title="US trade with Trinidad and Tobago, US$ millions")
plt.show()

---
# What you downloaded

`raw/_sources.csv` records every file, where it came from, and whether it was fresh or an
offline copy. Activity 5 reads `raw/` to build the dataset.

In [ ]:
pd.read_csv(SOURCES_CSV)

### Take-aways

| Shape of the source | Tool | Breaks when... |
|---|---|---|
| API | `requests` + `json` | the key expires |
| HTML table | `BeautifulSoup`, `pd.read_html` | the page layout changes |
| SDMX-XML | `ElementTree`, `{*}` wildcard | rarely: it is a standard |
| A file whose name changes | link harvesting | the page moves |
| Excel with merged headers | `header=None`, pattern-matched sheet | a sheet is renamed |

**Look before you code** (right-click → Inspect), **never hardcode what changes**, and
**check the result against something you know**.